In [3]:
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available GPUs:", torch.cuda.device_count())

base_dir = "/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"
train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")

Available GPUs: 2


In [7]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=val_transform)

targets = train_dataset.targets
class_counts = [targets.count(0), targets.count(1)]
class_weights = [sum(class_counts) / (2 * c) for c in class_counts]
sample_weights = [class_weights[label] for _, label in train_dataset]
sampler = WeightedRandomSampler(
    sample_weights, len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=64, sampler=sampler, num_workers=2, pin_memory=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

In [8]:
model_mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model_mobilenet.classifier[1] = nn.Linear(model_mobilenet.last_channel, 2)

if torch.cuda.device_count() > 1:
  model_mobilenet = nn.DataParallel(model_mobilenet)

model_mobilenet = model_mobilenet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_mobilenet = optim.AdamW(model_mobilenet.parameters(), lr=0.0001) 

In [10]:
model_mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model_mobilenet.classifier[1] = nn.Linear(model_mobilenet.last_channel, 2)

if torch.cuda.device_count() > 1:
  model_mobilenet = nn.DataParallel(model_mobilenet)

model_mobilenet = model_mobilenet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_mobilenet = optim.AdamW(model_mobilenet.parameters(), lr=0.0001)

In [11]:
epochs = 10

for epoch in range(epochs):
  model_mobilenet.train()
  train_loss = 0.0
  correct = 0
  total = 0

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer_mobilenet.zero_grad()
    outputs = model_mobilenet(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer_mobilenet.step()

    train_loss += loss.item()
    _, preds = torch.max(outputs, 1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

  train_acc = correct / total

  model_mobilenet.eval()
  val_loss = 0.0
  val_correct = 0
  val_total = 0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in valid_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model_mobilenet(images)
      loss = criterion(outputs, labels)
      val_loss += loss.item()
      _, preds = torch.max(outputs, 1)
      val_correct += (preds == labels).sum().item()
      val_total += labels.size(0)
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  val_acc = val_correct / val_total
  print(
      f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Train"
      f" Acc: {train_acc:.4f} | Val Loss: {val_loss/len(valid_loader):.4f} | Val"
      f" Acc: {val_acc:.4f}"
  )

# Save underlying model weights properly when using DataParallel
torch.save(
    model_mobilenet.module.state_dict()
    if isinstance(model_mobilenet, nn.DataParallel)
    else model_mobilenet.state_dict(),
    "/kaggle/working/mobilenetv2_wildfire.pth",
)
print(classification_report(all_labels, all_preds))

Epoch 1 | Train Loss: 0.1087 | Train Acc: 0.9637 | Val Loss: 0.0481 | Val Acc: 0.9841
Epoch 2 | Train Loss: 0.0346 | Train Acc: 0.9889 | Val Loss: 0.0346 | Val Acc: 0.9898
Epoch 3 | Train Loss: 0.0216 | Train Acc: 0.9929 | Val Loss: 0.0284 | Val Acc: 0.9905
Epoch 4 | Train Loss: 0.0131 | Train Acc: 0.9958 | Val Loss: 0.0333 | Val Acc: 0.9906
Epoch 5 | Train Loss: 0.0103 | Train Acc: 0.9968 | Val Loss: 0.0269 | Val Acc: 0.9914
Epoch 6 | Train Loss: 0.0111 | Train Acc: 0.9969 | Val Loss: 0.0328 | Val Acc: 0.9905
Epoch 7 | Train Loss: 0.0072 | Train Acc: 0.9979 | Val Loss: 0.0264 | Val Acc: 0.9927
Epoch 8 | Train Loss: 0.0061 | Train Acc: 0.9980 | Val Loss: 0.0266 | Val Acc: 0.9922
Epoch 9 | Train Loss: 0.0067 | Train Acc: 0.9979 | Val Loss: 0.0299 | Val Acc: 0.9914
Epoch 10 | Train Loss: 0.0047 | Train Acc: 0.9986 | Val Loss: 0.0295 | Val Acc: 0.9922
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2820
           1       1.00      0

In [12]:
model_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 2)

if torch.cuda.device_count() > 1:
  model_resnet = nn.DataParallel(model_resnet)

model_resnet = model_resnet.to(device)
optimizer_resnet = optim.AdamW(model_resnet.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s] 


In [13]:
for epoch in range(epochs):
  model_resnet.train()
  train_loss = 0.0
  correct = 0
  total = 0

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer_resnet.zero_grad()
    outputs = model_resnet(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer_resnet.step()

    train_loss += loss.item()
    _, preds = torch.max(outputs, 1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

  train_acc = correct / total

  model_resnet.eval()
  val_loss = 0.0
  val_correct = 0
  val_total = 0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in valid_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model_resnet(images)
      loss = criterion(outputs, labels)
      val_loss += loss.item()
      _, preds = torch.max(outputs, 1)
      val_correct += (preds == labels).sum().item()
      val_total += labels.size(0)
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  val_acc = val_correct / val_total
  print(
      f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Train"
      f" Acc: {train_acc:.4f} | Val Loss: {val_loss/len(valid_loader):.4f} | Val"
      f" Acc: {val_acc:.4f}"
  )

torch.save(
    model_resnet.module.state_dict()
    if isinstance(model_resnet, nn.DataParallel)
    else model_resnet.state_dict(),
    "/kaggle/working/resnet18_wildfire.pth",
)
print(classification_report(all_labels, all_preds))

Epoch 1 | Train Loss: 0.0643 | Train Acc: 0.9757 | Val Loss: 0.0395 | Val Acc: 0.9860
Epoch 2 | Train Loss: 0.0311 | Train Acc: 0.9894 | Val Loss: 0.0334 | Val Acc: 0.9895
Epoch 3 | Train Loss: 0.0187 | Train Acc: 0.9942 | Val Loss: 0.0237 | Val Acc: 0.9932
Epoch 4 | Train Loss: 0.0186 | Train Acc: 0.9939 | Val Loss: 0.0544 | Val Acc: 0.9832
Epoch 5 | Train Loss: 0.0143 | Train Acc: 0.9950 | Val Loss: 0.0268 | Val Acc: 0.9911
Epoch 6 | Train Loss: 0.0127 | Train Acc: 0.9957 | Val Loss: 0.0299 | Val Acc: 0.9905
Epoch 7 | Train Loss: 0.0133 | Train Acc: 0.9955 | Val Loss: 0.0721 | Val Acc: 0.9789
Epoch 8 | Train Loss: 0.0102 | Train Acc: 0.9964 | Val Loss: 0.0301 | Val Acc: 0.9908
Epoch 9 | Train Loss: 0.0095 | Train Acc: 0.9971 | Val Loss: 0.0343 | Val Acc: 0.9897
Epoch 10 | Train Loss: 0.0079 | Train Acc: 0.9974 | Val Loss: 0.0291 | Val Acc: 0.9894
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      2820
           1       0.99      0